In [2]:
import pandas as pd
import re
from collections import Counter

# 1. Load dataset

df = pd.read_csv("/content/sample_data/IMDB Dataset.csv")
print("Columns:", df.columns.tolist())
print(df.head())

# 2. Define stopwords

stopwords = set([
    'i','me','my','we','our','you','he','she','it','they',
    'the','a','an','and','but','or','if','to','of','in','on','for','with','as',
    'this','that','is','was','are','be','have','has','had','at','by','from',
    'so','not','can','will','all','one','like','film','movie','about','your',
    'myself','him','her','them','their','its','our','ours','yourself'
])


# 3. Split positive & negative reviews

positive_reviews = df[df['sentiment'] == 'positive']
negative_reviews = df[df['sentiment'] == 'negative']


# 4. Function to clean and tokenize review

def clean_review_words(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # remove punctuation
    words = text.split()
    words = [w for w in words if w not in stopwords and len(w) > 2]
    return words


# 5. Count keyword occurrences in all reviews

def keyword_counts(reviews, keyword_list):
    counts = Counter()
    for r in reviews['review']:
        words = clean_review_words(r)
        for k in keyword_list:
            counts[k] += words.count(k)
    return counts


# 6. Candidate keywords

all_keywords_candidate = [
    'great', 'amazing', 'excellent', 'wonderful', 'best', 'love', 'fun', 'perfect',
    'terrible', 'awful', 'worst', 'boring', 'bad', 'poor', 'hate', 'disappointing'
]

# Count occurrences

pos_counts = keyword_counts(positive_reviews, all_keywords_candidate)
neg_counts = keyword_counts(negative_reviews, all_keywords_candidate)


# 7. Select top 3 positive & negative keywords

top_positive = [word for word, _ in pos_counts.most_common(3)]
top_negative = [word for word, _ in neg_counts.most_common(3)]
print("Top Positive Keywords:", top_positive)
print("Top Negative Keywords:", top_negative)


# 8. Count number of reviews containing each keyword

def keyword_review_counts_all_clean(reviews, keywords):
    counts = {}
    for k in keywords:
        counts[k] = reviews['review'].apply(
            lambda x: k in clean_review_words(x)
        ).sum()
    return counts

all_top_keywords = top_positive + top_negative

pos_review_counts = keyword_review_counts_all_clean(positive_reviews, all_top_keywords)
neg_review_counts = keyword_review_counts_all_clean(negative_reviews, all_top_keywords)

# Convert np.int64 to regular int for clean printing

pos_review_counts = {k:int(v) for k,v in pos_review_counts.items()}
neg_review_counts = {k:int(v) for k,v in neg_review_counts.items()}

print("\nNumber of Positive Reviews containing each keyword:", pos_review_counts)
print("Number of Negative Reviews containing each keyword:", neg_review_counts)

Columns: ['review', 'sentiment']
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Top Positive Keywords: ['great', 'love', 'best']
Top Negative Keywords: ['bad', 'great', 'worst']

Number of Positive Reviews containing each keyword: {'great': 8478, 'love': 5666, 'best': 6239, 'bad': 2961, 'worst': 411}
Number of Negative Reviews containing each keyword: {'great': 4099, 'love': 3258, 'best': 3430, 'bad': 8827, 'worst': 4023}


In [9]:

# Bayes Probability Computation


# Combine top positive and negative keywords
keywords = top_positive + top_negative

# Prior probability
P_positive = len(positive_reviews) / len(df)

# Prepare list to store results
results = []

# Loop through each keyword
for k in keywords:
    # Number of reviews containing the keyword
    pos_count = pos_review_counts[k]
    print(pos_count)
    neg_count = neg_review_counts[k]
    total_count = pos_count + neg_count

    # Likelihood: P(keyword | Positive)
    likelihood = pos_count / len(positive_reviews)

    # Marginal: P(keyword)
    marginal = total_count / len(df)

    # Posterior: P(Positive | keyword) using Bayes theorem
    posterior = (likelihood * P_positive) / marginal if marginal != 0 else 0

    # Store results with rounded values for better readability
    results.append({
        'Keyword': k,
        'Prior P(Positive)': round(P_positive, 4),
        'Likelihood P(keyword|Positive)': round(likelihood, 4),
        'Marginal P(keyword)': round(marginal, 4),
        'Posterior P(Positive|keyword)': round(posterior, 4)
    })

# Convert results to DataFrame
bayes_table = pd.DataFrame(results)

# Display nicely in Jupyter (no backslashes)
bayes_table

8478
5666
6239
2961
8478
411


,Keyword,Prior P(Positive),Likelihood P(keyword|Positive),Marginal P(keyword),Posterior P(Positive|keyword)
0,great,0.5,0.3391,0.2515,0.6741
1,love,0.5,0.2266,0.1785,0.6349
2,best,0.5,0.2496,0.1934,0.6453
3,bad,0.5,0.1184,0.2358,0.2512
4,great,0.5,0.3391,0.2515,0.6741
5,worst,0.5,0.0164,0.0887,0.0927
